In [1]:
import pandas as pd
df = pd.read_csv("data/land_acquisition_dataset.csv")
df.head()

,project_id,state,district,project_type,land_required_hectares,land_acquired_hectares,pending_land_hectares,pct_land_acquired,pending_approvals_count,legal_disputes_count,compensation_delay_days,documentation_incomplete,rehabilitation_status,project_age_days,expected_duration_days,start_date,expected_completion_date,delay_probability,is_delayed
0,PRJ1000,Assam,Guwahati,Irrigation Canal,2350.1,278.0,2072.1,11.8,3,2,277,True,Completed,1462,745,21-09-2022,05-10-2024,45.6,0
1,PRJ1001,Rajasthan,Udaipur,Industrial Corridor,1567.0,31.3,1535.7,2.0,4,0,153,True,In Progress,109,1851,05-06-2026,30-06-2031,62.4,1
2,PRJ1002,Odisha,Cuttack,Urban Infrastructure,1584.2,210.2,1374.0,13.3,5,0,180,True,Pending,573,824,26-02-2025,31-05-2027,44.9,0
3,PRJ1003,Uttar Pradesh,Agra,Power Transmission Line,269.9,73.2,196.7,27.1,1,0,184,True,Completed,507,896,03-05-2025,16-10-2027,41.3,0
4,PRJ1004,Odisha,Cuttack,Irrigation Canal,1705.7,1139.3,566.4,66.8,1,0,90,False,Completed,1349,1142,12-01-2023,27-02-2026,12.8,0


In [2]:
print(df.shape)
print(df.info)
print(df.isnull().sum())

(2000, 19)
<bound method DataFrame.info of      project_id           state  district             project_type  \
0       PRJ1000           Assam  Guwahati         Irrigation Canal   
1       PRJ1001       Rajasthan   Udaipur      Industrial Corridor   
2       PRJ1002          Odisha   Cuttack     Urban Infrastructure   
3       PRJ1003   Uttar Pradesh      Agra  Power Transmission Line   
4       PRJ1004          Odisha   Cuttack         Irrigation Canal   
...         ...             ...       ...                      ...   
1995    PRJ2995     Maharashtra    Nashik     Urban Infrastructure   
1996    PRJ2996  Madhya Pradesh    Bhopal                  Highway   
1997    PRJ2997       Rajasthan   Jodhpur         Irrigation Canal   
1998    PRJ2998  Madhya Pradesh  Jabalpur         Railway Corridor   
1999    PRJ2999  Madhya Pradesh    Bhopal         Railway Corridor   

      land_required_hectares  land_acquired_hectares  pending_land_hectares  \
0                     2350.1         

In [3]:
df['is_delayed'].value_counts()

is_delayed
0    1669
1     331
Name: count, dtype: int64

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Columns jo model mein nahi jaani chahiye (IDs, dates, aur target khud)
drop_cols = ['project_id', 'start_date', 'expected_completion_date', 
             'delay_probability', 'is_delayed']

X = df.drop(columns=drop_cols)
y_class = df['is_delayed']
y_prob = df['delay_probability']

# Categorical columns ko number mein convert karo
categorical_cols = ['state', 'district', 'project_type', 'rehabilitation_status']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

# documentation_incomplete True/False ko 1/0 mein
X['documentation_incomplete'] = X['documentation_incomplete'].astype(int)

X.head()

,state,district,project_type,land_required_hectares,land_acquired_hectares,pending_land_hectares,pct_land_acquired,pending_approvals_count,legal_disputes_count,compensation_delay_days,documentation_incomplete,rehabilitation_status,project_age_days,expected_duration_days
0,0,10,2,2350.1,278.0,2072.1,11.8,3,2,277,1,0,1462,745
1,5,30,1,1567.0,31.3,1535.7,2.0,4,0,153,1,1,109,1851
2,4,6,5,1584.2,210.2,1374.0,13.3,5,0,180,1,2,573,824
3,6,0,3,269.9,73.2,196.7,27.1,1,0,184,1,0,507,896
4,4,6,2,1705.7,1139.3,566.4,66.8,1,0,90,0,0,1349,1142


In [5]:
X_train, X_test, y_class_train, y_class_test, y_prob_train, y_prob_test = train_test_split(
    X, y_class, y_prob, test_size=0.2, random_state=42, stratify=y_class
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (1600, 14)
Test size: (400, 14)


In [6]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error

# Classification model - predicts whether project will be delayed
clf_model = RandomForestClassifier(n_estimators=200, max_depth=10, 
                                     class_weight='balanced', random_state=42)
clf_model.fit(X_train, y_class_train)

# Regression model - predicts delay probability percentage
reg_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
reg_model.fit(X_train, y_prob_train)

print("Both models trained successfully!")

Both models trained successfully!


In [7]:
# Evaluate classification model
y_pred = clf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_class_test, y_pred))
print("\nClassification Report:\n", classification_report(y_class_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_class_test, y_pred))

# Evaluate regression model
y_prob_pred = reg_model.predict(X_test)
print("\nMean Absolute Error (probability):", mean_absolute_error(y_prob_test, y_prob_pred))

Accuracy: 0.8875

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.96      0.93       334
           1       0.72      0.52      0.60        66

    accuracy                           0.89       400
   macro avg       0.82      0.74      0.77       400
weighted avg       0.88      0.89      0.88       400


Confusion Matrix:
 [[321  13]
 [ 32  34]]

Mean Absolute Error (probability): 6.533893533812038


In [8]:
import pandas as pd

# Get feature importance from classification model
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': clf_model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(feature_importance)

                     feature  importance
10  documentation_incomplete    0.205541
9    compensation_delay_days    0.128041
7    pending_approvals_count    0.112925
6          pct_land_acquired    0.108949
8       legal_disputes_count    0.090008
4     land_acquired_hectares    0.070734
11     rehabilitation_status    0.064193
5      pending_land_hectares    0.046745
12          project_age_days    0.041516
3     land_required_hectares    0.035830
13    expected_duration_days    0.034241
1                   district    0.028660
0                      state    0.018086
2               project_type    0.014529


In [9]:
def get_top_reasons(row, feature_importance_df, top_n=3):
    # Multiply feature value's risk-relevance by global importance
    risk_features = ['pending_approvals_count', 'legal_disputes_count', 
                      'compensation_delay_days', 'documentation_incomplete', 
                      'pct_land_acquired']
    
    reasons = []
    if row['pending_approvals_count'] >= 4:
        reasons.append(('Pending approvals from concerned departments', row['pending_approvals_count']))
    if row['legal_disputes_count'] >= 2:
        reasons.append(('Active legal disputes affecting the project', row['legal_disputes_count']))
    if row['compensation_delay_days'] >= 150:
        reasons.append(('Compensation disbursement delayed', row['compensation_delay_days']))
    if row['documentation_incomplete'] == 1:
        reasons.append(('Incomplete documentation', 1))
    if row['pct_land_acquired'] < 40:
        reasons.append(('Slow land acquisition rate compared to timeline', row['pct_land_acquired']))
    
    # Sort by severity, return top N
    reasons.sort(key=lambda x: x[1], reverse=True)
    return [r[0] for r in reasons[:top_n]]

# Test on one project from test set
sample = X_test.iloc[0]
print(get_top_reasons(sample, feature_importance))

['Compensation disbursement delayed', 'Slow land acquisition rate compared to timeline', 'Pending approvals from concerned departments']


In [10]:
def predict_delay(row, clf_model, reg_model, feature_importance_df):
    # Prepare input in same format as training data
    input_data = row.values.reshape(1, -1)
    
    is_delayed_pred = clf_model.predict(input_data)[0]
    delay_prob_pred = reg_model.predict(input_data)[0]
    
    # Risk level based on probability
    if delay_prob_pred >= 60:
        risk_level = "High"
    elif delay_prob_pred >= 35:
        risk_level = "Medium"
    else:
        risk_level = "Low"
    
    reasons = get_top_reasons(row, feature_importance_df)
    
    return {
        "delay_probability": round(delay_prob_pred, 1),
        "is_delayed": bool(is_delayed_pred),
        "risk_level": risk_level,
        "top_reasons": reasons
    }

# Test on a sample project
sample = X_test.iloc[0]
result = predict_delay(sample, clf_model, reg_model, feature_importance)
print(result)

{'delay_probability': np.float64(63.0), 'is_delayed': True, 'risk_level': 'High', 'top_reasons': ['Compensation disbursement delayed', 'Slow land acquisition rate compared to timeline', 'Pending approvals from concerned departments']}


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [11]:
import pickle

with open('delay_model.pkl', 'wb') as f:
    pickle.dump({'classifier': clf_model, 'regressor': reg_model, 
                 'label_encoders': label_encoders, 'feature_importance': feature_importance}, f)

print("Model saved successfully!")

Model saved successfully!
